# Netherlands (RDT) — Preprocessing notebook

**Period:** 2024-01-01 → 2024-06-30

**Raw inputs** (`Data/Netherlands/Raw/`):
- `services-2024-{01..06}.csv` — six monthly RDT exports, stop-level, with `Service:` / `Stop:` colon-prefixed columns
- `stations-2023-09.csv` — station master (`code`, `geo_lat`, `geo_lng`)
- `tariff-distances-2022-01.csv` — wide station × station distance matrix
- `disruptions-2024.csv` — disruption events with comma-separated affected stations
- + Open-Meteo enrichment (free historical archive API)

**Outputs** (`Data/Netherlands/processed/`): the five standardized CSVs of `utils.SCHEMA` plus the cached `weather_by_station.parquet`.

Heavy logic lives in `preprocess/lib_netherlands.py`; the Open-Meteo client is in `weather_enrichment_nl.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd().resolve()
while ROOT.name and not (ROOT / "utils.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "preprocess"))

import lib_netherlands as lib
from utils import SCHEMA, save_csv

RAW = ROOT / "Data" / "Netherlands" / "Raw"
OUT = ROOT / "Data" / "Netherlands" / "processed"
FIG = ROOT / "preprocess" / "figures" / "netherlands"
OUT.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
np.random.seed(42)

def save_fig(fig, name: str):
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight")

## Step 1 — Glob and load six monthly files

`lib.load_services` does the heavy lifting: rename `Service:Foo` / `Stop:Foo` columns to snake_case, parse dates, clean delay sentinels (`N` / `S` like Italy), build the union-of-three cancellation flag, and apply the date filter.

In [ ]:
df, file_log = lib.load_services(RAW)
print(f"NL stops kept: {len(df):,}")
file_log

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(file_log["file"].str[-10:-4], file_log["rows"], color="#FF6F00")
axes[0].set_title("Rows per monthly file"); axes[0].set_ylabel("rows")
axes[0].tick_params(axis="x", rotation=30)
axes[1].plot(file_log["file"].str[-10:-4], file_log["rows"].cumsum(),
             marker="o", color="#FF6F00")
axes[1].set_title("Cumulative rows loaded"); axes[1].tick_params(axis="x", rotation=30)
fig.suptitle("Step 1 — File loading", fontweight="bold")
save_fig(fig, "step_01_loading"); plt.show()

## Step 2 — Column rename map

Reference table for the `Service:Foo` → snake_case rename applied at load time.

In [ ]:
rename_df = (
    pd.Series(lib.NL_RENAME, name="renamed").reset_index()
      .rename(columns={"index": "original"})
)
rename_df

## Step 3 — Date filter

Histogram of `date` post-filter.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(df["date"].dropna(), bins=24, color="#FF6F00", edgecolor="white")
ax.set_title("Step 3 — date distribution after [2024-01-01, 2024-06-30] filter",
             fontweight="bold"); ax.set_xlabel("date")
save_fig(fig, "step_03_date_filter"); plt.show()

## Step 4 — Cancellation flag breakdown

Three independent boolean sources combine into the canonical `cancelled` field. The bar chart shows how many rows each source contributes — useful for catching the case where a single source dominates.

In [ ]:
cancel_break = lib.cancellation_breakdown(df)
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = ["#90A4AE"] * (len(cancel_break) - 1) + ["#E53935"]
ax.bar(cancel_break["source"], cancel_break["n_true"], color=colors)
ax.set_title("Step 4 — Cancellation flag sources and their union", fontweight="bold")
ax.set_ylabel("# rows = True"); ax.tick_params(axis="x", rotation=20)
save_fig(fig, "step_04_cancellation"); plt.show()
cancel_break

## Step 5 — Open-Meteo weather enrichment

We hit the Open-Meteo Historical Archive API once per station (full date range) and cache responses in `Data/Netherlands/processed/.weather_cache/` via `requests-cache`. The notebook reports the API/cache outcome and the per-station completeness.

In [ ]:
stations_master = pd.read_csv(RAW / "stations-2023-09.csv")

weather_path = OUT / "weather_by_station.parquet"
if weather_path.exists():
    weather_by_station = pd.read_parquet(weather_path)
    print(f"Loaded cached enrichment: {weather_by_station.shape}")
else:
    from weather_enrichment_nl import enrich_nl_with_weather
    df = enrich_nl_with_weather(df, stations_master)
    weather_by_station = pd.read_parquet(weather_path) if weather_path.exists() else df
df.head(3)

In [ ]:
weather_cols = [c for c in ["temperature", "wind_speed", "precipitation",
                              "snow_depth", "weather_severity"] if c in df.columns]
completeness = (1.0 - df[weather_cols].isna().mean()).to_frame("completeness")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].barh(completeness.index, completeness["completeness"], color="#FF6F00")
axes[0].set_xlim(0, 1); axes[0].set_title("Completeness per weather column")
if "temperature" in df.columns:
    axes[1].hist(df["temperature"].dropna(), bins=60, color="#FF6F00", edgecolor="white")
axes[1].set_title("Temperature distribution"); axes[1].set_xlabel("°C")
fig.suptitle("Step 5 — Open-Meteo enrichment", fontweight="bold")
save_fig(fig, "step_05_openmeteo"); plt.show()

## Step 6 — WMO code → severity ordinal

If the enrichment produced `weather_code` (the WMO integer), we project it to a 0–4 ordinal for parity with Italy and Finland.

In [ ]:
if "weather_severity" in df.columns:
    sev_counts = df["weather_severity"].value_counts().sort_index()
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.bar(sev_counts.index.astype(str), sev_counts.values,
            color=sns.color_palette("Oranges", n_colors=5))
    ax.set_title("Step 6 — Severity ordinal counts", fontweight="bold")
    ax.set_xlabel("severity 0–4"); ax.set_ylabel("row count")
    save_fig(fig, "step_06_severity"); plt.show()
else:
    print("No weather_severity column — skipping severity plot.")

## Step 7 — NL station map

In [ ]:
nodes_station = lib.build_nodes_station(df, stations_master)
geo = nodes_station.dropna(subset=["lat", "lon"])

fig, ax = plt.subplots(figsize=(7, 8))
sc = ax.scatter(geo["lon"], geo["lat"],
                c=geo["avg_historical_delay"].clip(0, 5),
                s=10 + geo["degree"].clip(upper=200) / 10,
                cmap="Reds", edgecolor="k", linewidth=0.2)
plt.colorbar(sc, ax=ax, label="avg delay (min)")
ax.set_title("Step 7 — Netherlands station map\n(size = degree, colour = avg delay)",
             fontweight="bold")
ax.set_xlabel("lon"); ax.set_ylabel("lat")
save_fig(fig, "step_07_station_map"); plt.show()

## Step 8 — Tariff-distance matrix → adjacency edges

The raw distance file is **wide** (station × station). We replace `'XXX'` (diagonal sentinel) with NaN, melt to long format, and keep only edges where both endpoints map to known NL stations.

In [ ]:
sid_map = {c: f"NL_{c}" for c in df["station_code"].dropna().unique()}
edges_adjacent, info = lib.build_edges_adjacent(
    RAW / "tariff-distances-2022-01.csv", sid_map)
info

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].pie([info["n_xxx_diag"], info["n_nan"] - info["n_xxx_diag"],
             info["n_total"] - info["n_nan"]],
            labels=["XXX (diag)", "NaN", "valid km"],
            colors=["#90A4AE", "#E0E0E0", "#FF6F00"],
            autopct="%1.1f%%")
axes[0].set_title("Tariff matrix sparsity")
axes[1].hist(edges_adjacent["distance_km"], bins=60, color="#FF6F00", edgecolor="white")
axes[1].set_title("Edge distance_km distribution"); axes[1].set_xlabel("km")
assert (edges_adjacent["distance_km"].dropna() >= 0).all(), "negative distances!"
fig.suptitle("Step 8 — Tariff-distance edges", fontweight="bold")
save_fig(fig, "step_08_tariff"); plt.show()

## Step 9 — Disruption log → fault nodes

Each disruption row carries a comma-separated list of affected station codes. We explode to one fault row per (disruption × affected station). The bar chart shows the top-20 root causes.

In [ ]:
disruptions = pd.read_csv(RAW / "disruptions-2024.csv")
nodes_fault, cause_top = lib.build_nodes_fault(disruptions, sid_map)
print(f"fault rows after explode: {len(nodes_fault):,}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(cause_top["cause_en"][::-1], cause_top["n_faults"][::-1], color="#E53935")
ax.set_title("Step 9 — Top-20 disruption causes", fontweight="bold")
ax.set_xlabel("# unique fault events")
save_fig(fig, "step_09_causes"); plt.show()

## Step 10 — Service labelling

In [ ]:
nodes_service = lib.build_nodes_service(df)
print(f"services: {len(nodes_service):,} | disruption rate = {nodes_service['is_disrupted'].mean()*100:.2f}%")

by_class = nodes_service.groupby("train_class_code")["is_disrupted"].mean().reset_index()
by_month = (
    nodes_service.assign(m=pd.to_datetime(nodes_service["date"]).dt.month)
                  .groupby("m")["is_disrupted"].mean()
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(by_class["train_class_code"].astype(str), by_class["is_disrupted"], color="#FF6F00")
axes[0].set_title("by train class"); axes[0].set_ylim(0, 1)
axes[1].plot(by_month.index, by_month.values, marker="o", color="#E91E63")
axes[1].set_title("by month"); axes[1].set_xticks(range(1, 7)); axes[1].set_ylim(0, 1)
fig.suptitle("Step 10 — Service-level disruption rates", fontweight="bold")
save_fig(fig, "step_10_service_rates"); plt.show()

## Step 11 — Persist all five standardized CSVs

In [ ]:
edges_stops_at = lib.build_edges_stops_at(df, sid_map)

save_csv(nodes_station,  OUT / "nodes_station.csv",  "nodes_station")
save_csv(nodes_service,  OUT / "nodes_service.csv",  "nodes_service")
save_csv(edges_stops_at, OUT / "edges_stops_at.csv", "edges_stops_at")
save_csv(edges_adjacent, OUT / "edges_adjacent.csv", "edges_adjacent")
save_csv(nodes_fault,    OUT / "nodes_fault.csv",    "nodes_fault")

summary = pd.DataFrame([
    {"file": "nodes_station.csv",  "rows": len(nodes_station)},
    {"file": "nodes_service.csv",  "rows": len(nodes_service)},
    {"file": "edges_stops_at.csv", "rows": len(edges_stops_at)},
    {"file": "edges_adjacent.csv", "rows": len(edges_adjacent)},
    {"file": "nodes_fault.csv",    "rows": len(nodes_fault)},
])
summary

In [ ]:
for name, dfx in [("nodes_station", nodes_station), ("nodes_service", nodes_service),
                   ("edges_stops_at", edges_stops_at), ("edges_adjacent", edges_adjacent),
                   ("nodes_fault", nodes_fault)]:
    expected = set(SCHEMA[name])
    actual   = set(dfx.columns)
    assert expected.issubset(actual) or len(dfx) == 0, f"{name}: missing {expected - actual}"
print("✓ Schema validation passed for all five outputs.")

## Closing summary

Netherlands preprocessing complete. Outputs land in `Data/Netherlands/processed/` (incl. `weather_by_station.parquet`); inspection plots in `preprocess/figures/netherlands/`. The unified stop-level pipeline consumes these next.